# 1) imports

In [1]:
from pathlib import Path
import json
import re
import unicodedata
from dataclasses import dataclass, asdict

import numpy as np
import pandas as pd

# 2) chemins

In [2]:
BASE_DIR = Path.cwd()

# Si le notebook est ouvert depuis Assistant_IA/notebooks
if BASE_DIR.name == "notebooks":
    ASSISTANT_DIR = BASE_DIR.parent
#  depuis Assistant_IA
elif BASE_DIR.name == "Assistant_IA":
    ASSISTANT_DIR = BASE_DIR
else:
    ASSISTANT_DIR = BASE_DIR / "Assistant_IA"

DOCS_DIR = ASSISTANT_DIR / "docs_site"
INDEX_DIR = ASSISTANT_DIR / "data" / "site_rag_index"

INDEX_DIR.mkdir(parents=True, exist_ok=True)

print("ASSISTANT_DIR =", ASSISTANT_DIR)
print("DOCS_DIR =", DOCS_DIR)
print("INDEX_DIR =", INDEX_DIR)
print("docs_site exists ?", DOCS_DIR.exists())

ASSISTANT_DIR = c:\Users\Innocent\OneDrive\Bureau\@ProgrammeAI_cours_session_04\Projet Capstone en IA\CityTaste\Assistant_IA
DOCS_DIR = c:\Users\Innocent\OneDrive\Bureau\@ProgrammeAI_cours_session_04\Projet Capstone en IA\CityTaste\Assistant_IA\docs_site
INDEX_DIR = c:\Users\Innocent\OneDrive\Bureau\@ProgrammeAI_cours_session_04\Projet Capstone en IA\CityTaste\Assistant_IA\data\site_rag_index
docs_site exists ? True


# 3) lister les fichiers

In [3]:
doc_paths = sorted(DOCS_DIR.glob("*.md"))
[p.name for p in doc_paths]

['details_lieu.md',
 'distance.md',
 'faq.md',
 'filtres.md',
 'geolocalisation.md',
 'image.md',
 'recommandation.md',
 'resultats.md',
 'site_policy.md']

# 4) lecture des documents

In [4]:
documents = []

for path in doc_paths:
    text = path.read_text(encoding="utf-8").strip()
    documents.append({
        "source_file": path.name,
        "title": path.stem,
        "text": text
    })

len(documents), [d["source_file"] for d in documents]

(9,
 ['details_lieu.md',
  'distance.md',
  'faq.md',
  'filtres.md',
  'geolocalisation.md',
  'image.md',
  'recommandation.md',
  'resultats.md',
  'site_policy.md'])

# 5) Aperçu rapide

In [5]:
for d in documents:
    print("=" * 80)
    print(d["source_file"])
    print(d["text"][:500], "...\n")

details_lieu.md
# Détails d’un lieu — Comprendre la fiche détaillée

## À quoi sert la fiche détaillée ?

La fiche détaillée sert à afficher plus d’informations sur un lieu sélectionné dans les résultats. Elle aide l’utilisateur à mieux comprendre l’endroit avant de décider s’il veut le visiter, le comparer ou consulter son site web.

La fiche détaillée est particulièrement utile quand l’utilisateur veut aller au-delà d’une simple carte de résultat.

---

## Quelles informations peut-on trouver dans une fiche d ...

distance.md
# Distance — Comment l’interpréter dans CityTaste

## Que signifie la distance affichée ?

La distance affichée dans CityTaste sert à donner une **idée de proximité** entre un lieu et un point de référence utilisé par l’application.

Dans une version où la position utilisateur n’est pas activée, la distance correspond généralement à une **distance calculée depuis une référence centrale**, par exemple le centre d’Ottawa. Elle ne représente donc pas automatiquemen

# 6) Fonctions de nettoyage

In [6]:
def normalize_text(text: str) -> str:
    text = "" if text is None else str(text)
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def clean_markdown_text(text: str) -> str:
    text = normalize_text(text)

    # enlève espaces multiples
    text = re.sub(r"[ \t]+", " ", text)
    # garde les sauts de ligne utiles
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()

# 7) Re-nettoyage les documents

In [7]:
for d in documents:
    d["text"] = clean_markdown_text(d["text"])

documents[0]["text"][:1000]

'# Détails d’un lieu — Comprendre la fiche détaillée\n\n## À quoi sert la fiche détaillée ?\n\nLa fiche détaillée sert à afficher plus d’informations sur un lieu sélectionné dans les résultats. Elle aide l’utilisateur à mieux comprendre l’endroit avant de décider s’il veut le visiter, le comparer ou consulter son site web.\n\nLa fiche détaillée est particulièrement utile quand l’utilisateur veut aller au-delà d’une simple carte de résultat.\n\n---\n\n## Quelles informations peut-on trouver dans une fiche détaillée ?\n\nSelon les données disponibles, une fiche détaillée peut contenir :\n\n- le nom du lieu\n- le type d’établissement\n- la cuisine pour les restaurants\n- l’adresse\n- les horaires\n- le site web\n- certaines informations d’accessibilité\n- la note et le nombre d’avis lorsqu’ils existent\n- une image si elle est disponible\n- un texte descriptif ou enrichi\n\nToutes les fiches n’ont pas exactement le même niveau de détail.\n\n---\n\n## Pourquoi certaines informations manque

# 8) Structure d'un chunk

In [8]:
@dataclass
class SiteChunk:
    chunk_id: str
    source_file: str
    title: str
    section_title: str
    text: str
    char_len: int

# 9) Découpage intelligent par sections markdown

In [9]:
# On ne veut pas juste couper au hasard.
# On veut garder les sections ##, et parce que ce sont e;;e qui portent le sens.
def split_markdown_into_sections(doc_title: str, source_file: str, text: str):
    """
    Découpe un markdown en sections basées sur les titres ##.
    Si aucun ## n'existe, retourne un seul bloc.
    """
    lines = text.split("\n")
    sections = []

    current_title = "Introduction"
    current_lines = []

    for line in lines:
        if line.startswith("## "):
            if current_lines:
                sections.append({
                    "source_file": source_file,
                    "title": doc_title,
                    "section_title": current_title,
                    "text": "\n".join(current_lines).strip()
                })
            current_title = line.replace("## ", "").strip()
            current_lines = [line]
        else:
            current_lines.append(line)

    if current_lines:
        sections.append({
            "source_file": source_file,
            "title": doc_title,
            "section_title": current_title,
            "text": "\n".join(current_lines).strip()
        })

    return sections

### 10) Créer les sections

In [10]:
sections = []

for doc in documents:
    doc_sections = split_markdown_into_sections(
        doc_title=doc["title"],
        source_file=doc["source_file"],
        text=doc["text"]
    )
    sections.extend(doc_sections)

len(sections)

81

### 11) Aperçu des sections

In [11]:
sections_df = pd.DataFrame(sections)
sections_df[["source_file", "section_title"]].head(20)

,source_file,section_title
0,details_lieu.md,Introduction
1,details_lieu.md,À quoi sert la fiche détaillée ?
2,details_lieu.md,Quelles informations peut-on trouver dans une ...
3,details_lieu.md,Pourquoi certaines informations manquent dans ...
4,details_lieu.md,Comment ouvrir la fiche d’un lieu ?
5,details_lieu.md,La fiche détaillée suffit-elle pour prendre un...
6,details_lieu.md,Pourquoi deux fiches détaillées n’ont-elles pa...
7,details_lieu.md,Formulations fréquentes
8,distance.md,Introduction
9,distance.md,Que signifie la distance affichée ?


### 12) Chunking final

In [12]:
def split_long_text(text: str, max_chars: int = 700, overlap: int = 120):
    """
    Découpe un texte long en sous-blocs avec overlap.
    """
    text = text.strip()

    if len(text) <= max_chars:
        return [text]

    chunks = []
    start = 0

    while start < len(text):
        end = start + max_chars
        chunk = text[start:end]

        # essayer de couper à la fin d'une phrase
        if end < len(text):
            last_break = max(
                chunk.rfind(". "),
                chunk.rfind("\n"),
                chunk.rfind("? "),
                chunk.rfind("! ")
            )
            if last_break > int(max_chars * 0.5):
                end = start + last_break + 1
                chunk = text[start:end]

        chunks.append(chunk.strip())

        if end >= len(text):
            break

        start = max(0, end - overlap)

    return chunks

### 13) COnstruiction des chunks finaux

In [13]:
# Construiction des chunks finaux
chunks = []

for sec_idx, sec in enumerate(sections):
    small_parts = split_long_text(sec["text"], max_chars=700, overlap=120)

    for part_idx, part in enumerate(small_parts):
        chunk = SiteChunk(
            chunk_id=f"{sec['title']}_{sec_idx}_{part_idx}",
            source_file=sec["source_file"],
            title=sec["title"],
            section_title=sec["section_title"],
            text=part,
            char_len=len(part)
        )
        chunks.append(asdict(chunk))

len(chunks)

81

### 14) Vérification des chunks

In [14]:
chunks_df = pd.DataFrame(chunks)
chunks_df.head(10)

,chunk_id,source_file,title,section_title,text,char_len
0,details_lieu_0_0,details_lieu.md,details_lieu,Introduction,# Détails d’un lieu — Comprendre la fiche déta...,51
1,details_lieu_1_0,details_lieu.md,details_lieu,À quoi sert la fiche détaillée ?,## À quoi sert la fiche détaillée ?\n\nLa fich...,389
2,details_lieu_2_0,details_lieu.md,details_lieu,Quelles informations peut-on trouver dans une ...,## Quelles informations peut-on trouver dans u...,483
3,details_lieu_3_0,details_lieu.md,details_lieu,Pourquoi certaines informations manquent dans ...,## Pourquoi certaines informations manquent da...,405
4,details_lieu_4_0,details_lieu.md,details_lieu,Comment ouvrir la fiche d’un lieu ?,## Comment ouvrir la fiche d’un lieu ?\n\nEn g...,288
5,details_lieu_5_0,details_lieu.md,details_lieu,La fiche détaillée suffit-elle pour prendre un...,## La fiche détaillée suffit-elle pour prendre...,394
6,details_lieu_6_0,details_lieu.md,details_lieu,Pourquoi deux fiches détaillées n’ont-elles pa...,## Pourquoi deux fiches détaillées n’ont-elles...,466
7,details_lieu_7_0,details_lieu.md,details_lieu,Formulations fréquentes,## Formulations fréquentes\n\nL’utilisateur pe...,393
8,distance_8_0,distance.md,distance,Introduction,# Distance — Comment l’interpréter dans CityTaste,49
9,distance_9_0,distance.md,distance,Que signifie la distance affichée ?,## Que signifie la distance affichée ?\n\nLa d...,494


### 15) Voir quelques exemples

In [15]:
for c in chunks[:5]:
    print("=" * 100)
    print("chunk_id:", c["chunk_id"])
    print("source_file:", c["source_file"])
    print("section_title:", c["section_title"])
    print("text:\n", c["text"][:800], "\n")

chunk_id: details_lieu_0_0
source_file: details_lieu.md
section_title: Introduction
text:
 # Détails d’un lieu — Comprendre la fiche détaillée 

chunk_id: details_lieu_1_0
source_file: details_lieu.md
section_title: À quoi sert la fiche détaillée ?
text:
 ## À quoi sert la fiche détaillée ?

La fiche détaillée sert à afficher plus d’informations sur un lieu sélectionné dans les résultats. Elle aide l’utilisateur à mieux comprendre l’endroit avant de décider s’il veut le visiter, le comparer ou consulter son site web.

La fiche détaillée est particulièrement utile quand l’utilisateur veut aller au-delà d’une simple carte de résultat.

--- 

chunk_id: details_lieu_2_0
source_file: details_lieu.md
section_title: Quelles informations peut-on trouver dans une fiche détaillée ?
text:
 ## Quelles informations peut-on trouver dans une fiche détaillée ?

Selon les données disponibles, une fiche détaillée peut contenir :

- le nom du lieu
- le type d’établissement
- la cuisine pour les restauran

### 16) Installation & importation de modèle d'embeddings

In [16]:
import numpy
print("numpy version:", numpy.__version__)
import pkg_resources
print("pkg_resources numpy version:", pkg_resources.get_distribution('numpy').version)
import importlib.metadata
print("importlib.metadata numpy version:", importlib.metadata.version('numpy'))
from sentence_transformers import SentenceTransformer

numpy version: 2.4.4


C:\Users\Innocent\AppData\Local\Temp\ipykernel_17080\3747748102.py:3: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


pkg_resources numpy version: 2.4.4
importlib.metadata numpy version: 2.4.4


c:\Users\Innocent\OneDrive\Bureau\@ProgrammeAI_cours_session_04\Projet Capstone en IA\CityTaste\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 17) Chargement du modèle

In [17]:
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

EMBEDDING_MODEL_NAME

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3029.67it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


'sentence-transformers/all-MiniLM-L6-v2'

### 18) Préparation des textes a encoder

In [18]:
!pip install --force-reinstall --no-deps numpy==2.4.2
!pip install --upgrade transformers

  Using cached numpy-2.4.2-cp311-cp311-win_amd64.whl.metadata (6.6 kB)
Using cached numpy-2.4.2-cp311-cp311-win_amd64.whl (12.6 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.4.4
    Uninstalling numpy-2.4.4:
      Successfully uninstalled numpy-2.4.4


  You can safely remove it manually.
  You can safely remove it manually.


   ---------------------------------------- 0.0/10.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/10.2 MB ? eta -:--:--
   --- ------------------------------------ 0.8/10.2 MB 3.7 MB/s eta 0:00:03
   ----- ---------------------------------- 1.3/10.2 MB 3.0 MB/s eta 0:00:03
   ------- -------------------------------- 1.8/10.2 MB 3.0 MB/s eta 0:00:03
   ---------- ----------------------------- 2.6/10.2 MB 3.2 MB/s eta 0:00:03
   ------------- -------------------------- 3.4/10.2 MB 3.4 MB/s eta 0:00:03
   ---------------- ----------------------- 4.2/10.2 MB 3.4 MB/s eta 0:00:02
   ------------------- -------------------- 5.0/10.2 MB 3.5 MB/s eta 0:00:02
   ---------------------- ----------------- 5.8/10.2 MB 3.5 MB/s eta 0:00:02
   ------------------------- -------------- 6.6/10.2 MB 3.5 MB/s eta 0:00:02
   ----------------------------- ---------- 7.6/10.2 MB 3.5 MB/s eta 0:00:01
   -------------------------------- ------- 8.4/10.2 MB 3.5 MB/s eta 0:00:01
   ----------

In [19]:
def build_chunk_for_embedding(chunk: dict) -> str:
    return (
        f"Titre document: {chunk['title']}\n"
        f"Section: {chunk['section_title']}\n"
        f"Contenu: {chunk['text']}"
    )

chunk_texts = [build_chunk_for_embedding(c) for c in chunks]
len(chunk_texts), chunk_texts[0][:500]

(81,
 'Titre document: details_lieu\nSection: Introduction\nContenu: # Détails d’un lieu — Comprendre la fiche détaillée')

### 19) Encoder les chunks

In [20]:
embeddings = embedding_model.encode(
    chunk_texts,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

embeddings.shape

Batches: 100%|██████████| 3/3 [00:06<00:00,  2.13s/it]


(81, 384)

## 20) Index de recherche

In [21]:
# Ici on utilise sklearn pour que ça marche facilement sur la machine.
from sklearn.neighbors import NearestNeighbors
import joblib

nn_index = NearestNeighbors(
    n_neighbors=5,
    metric="cosine"
)

nn_index.fit(embeddings)
print("Index prêt.")

Index prêt.


### 21) Fonction de recherche

In [22]:
def search_site_docs(query: str, top_k: int = 3):
    query_vec = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    distances, indices = nn_index.kneighbors(query_vec, n_neighbors=top_k)

    results = []
    for rank, (idx, dist) in enumerate(zip(indices[0], distances[0]), start=1):
        chunk = chunks[idx].copy()
        chunk["rank"] = rank
        chunk["distance"] = float(dist)
        chunk["similarity"] = float(1 - dist)
        results.append(chunk)

    return results

### 22) Texter la recherche

In [23]:
queries = [
    "Est-ce que je suis obligé d'activer ma localisation ?",
    "La distance est calculée par rapport à quoi ?",
    "Pourquoi certains lieux n'ont pas d'image ?",
    "Comment voir les détails d'un lieu ?",
    "Est-ce que CityTaste fait des réservations ?",
]

for q in queries:
    print("=" * 120)
    print("QUESTION :", q)
    results = search_site_docs(q, top_k=3)

    for r in results:
        print(f"\n[{r['rank']}] {r['source_file']} | {r['section_title']} | sim={r['similarity']:.4f}")
        print(r["text"][:500], "...")

QUESTION : Est-ce que je suis obligé d'activer ma localisation ?

[1] geolocalisation.md | Est-ce que l’utilisateur est obligé d’activer sa localisation ? | sim=0.6504
## Est-ce que l’utilisateur est obligé d’activer sa localisation ?

Non. L’activation de la localisation n’est pas obligatoire pour utiliser CityTaste.

L’utilisateur peut continuer à naviguer sur le site, consulter les résultats et explorer les lieux même s’il refuse l’accès à sa position. Le site reste utilisable sans géolocalisation.

--- ...

[2] geolocalisation.md | Formulations fréquentes | sim=0.6031
## Formulations fréquentes

L’utilisateur peut poser des questions comme :

- Est-ce que je suis obligé d’activer ma localisation ?
- Est-ce que le site marche sans ma position ?
- Que se passe-t-il si je refuse l’accès à ma position ?
- Pourquoi le site me demande ma localisation ?
- Est-ce que vous avez besoin de ma position pour trouver des lieux ?
- Puis-je utiliser CityTaste sans géolocalisation ? ...

[3] geoloc

### 23) Petite fonction de réponse sans LLM

In [24]:
def answer_site_question_basic(query: str, top_k: int = 3):
    results = search_site_docs(query, top_k=top_k)

    if not results:
        return {
            "question": query,
            "answer": "Je n’ai pas trouvé d’information pertinente dans la documentation du site.",
            "sources": []
        }

    best = results[0]

    return {
        "question": query,
        "answer": best["text"],
        "sources": [
            {
                "source_file": r["source_file"],
                "section_title": r["section_title"],
                "similarity": r["similarity"]
            }
            for r in results
        ]
    }

### 24) Tester la réponse basique

In [25]:
test_questions = [
    "Est-ce que je suis obligé d'activer ma localisation ?",
    "Que se passe-t-il si je refuse ma position ?",
    "Le site réserve-t-il à ma place ?",
    "Pourquoi certaines fiches n'ont pas de photo ?",
]

for q in test_questions:
    print("=" * 120)
    response = answer_site_question_basic(q, top_k=3)
    print("QUESTION :", response["question"])
    print("RÉPONSE :\n", response["answer"])
    print("\nSOURCES :", response["sources"])

QUESTION : Est-ce que je suis obligé d'activer ma localisation ?
RÉPONSE :
 ## Est-ce que l’utilisateur est obligé d’activer sa localisation ?

Non. L’activation de la localisation n’est pas obligatoire pour utiliser CityTaste.

L’utilisateur peut continuer à naviguer sur le site, consulter les résultats et explorer les lieux même s’il refuse l’accès à sa position. Le site reste utilisable sans géolocalisation.

---

SOURCES : [{'source_file': 'geolocalisation.md', 'section_title': 'Est-ce que l’utilisateur est obligé d’activer sa localisation ?', 'similarity': 0.6503698825836182}, {'source_file': 'geolocalisation.md', 'section_title': 'Formulations fréquentes', 'similarity': 0.6031413078308105}, {'source_file': 'geolocalisation.md', 'section_title': 'À quoi sert la localisation ?', 'similarity': 0.577049195766449}]
QUESTION : Que se passe-t-il si je refuse ma position ?
RÉPONSE :
 ## Que se passe-t-il si l’utilisateur refuse la géolocalisation ?

Si l’utilisateur refuse l’accès à sa p

### 25) Sauvegarder les chunks

In [26]:
CHUNKS_PATH = INDEX_DIR / "chunks.json"

with open(CHUNKS_PATH, "w", encoding="utf-8") as f:
    json.dump(chunks, f, ensure_ascii=False, indent=2)

print("Chunks sauvegardés :", CHUNKS_PATH)

Chunks sauvegardés : c:\Users\Innocent\OneDrive\Bureau\@ProgrammeAI_cours_session_04\Projet Capstone en IA\CityTaste\Assistant_IA\data\site_rag_index\chunks.json


### 26) Sauvegarder les embeddings

In [27]:
EMBEDDINGS_PATH = INDEX_DIR / "embeddings.npy"
np.save(EMBEDDINGS_PATH, embeddings)

print("Embeddings sauvegardés :", EMBEDDINGS_PATH)

Embeddings sauvegardés : c:\Users\Innocent\OneDrive\Bureau\@ProgrammeAI_cours_session_04\Projet Capstone en IA\CityTaste\Assistant_IA\data\site_rag_index\embeddings.npy


### 27) Sauvegarder l'index sklearn

In [28]:
INDEX_PATH = INDEX_DIR / "nn_index.joblib"
joblib.dump(nn_index, INDEX_PATH)

print("Index sauvegardé :", INDEX_PATH)

Index sauvegardé : c:\Users\Innocent\OneDrive\Bureau\@ProgrammeAI_cours_session_04\Projet Capstone en IA\CityTaste\Assistant_IA\data\site_rag_index\nn_index.joblib


### 28) Sauvegarder les métadonnées

In [29]:
METADATA_PATH = INDEX_DIR / "metadata.json"

metadata = {
    "embedding_model_name": EMBEDDING_MODEL_NAME,
    "num_docs": len(documents),
    "num_sections": len(sections),
    "num_chunks": len(chunks),
    "embedding_dim": int(embeddings.shape[1]),
    "index_type": "sklearn_nearest_neighbors_cosine"
}

with open(METADATA_PATH, "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

metadata

{'embedding_model_name': 'sentence-transformers/all-MiniLM-L6-v2',
 'num_docs': 9,
 'num_sections': 81,
 'num_chunks': 81,
 'embedding_dim': 384,
 'index_type': 'sklearn_nearest_neighbors_cosine'}

In [30]:
print("INDEX_DIR:", INDEX_DIR)
print("chunks.json exists ?", CHUNKS_PATH.exists())
print("embeddings.npy exists ?", EMBEDDINGS_PATH.exists())
print("nn_index.joblib exists ?", INDEX_PATH.exists())
print("metadata.json exists ?", METADATA_PATH.exists())

INDEX_DIR: c:\Users\Innocent\OneDrive\Bureau\@ProgrammeAI_cours_session_04\Projet Capstone en IA\CityTaste\Assistant_IA\data\site_rag_index
chunks.json exists ? True
embeddings.npy exists ? True
nn_index.joblib exists ? True
metadata.json exists ? True


In [31]:
import sys, sklearn
print(sys.executable)
print(sklearn.__version__)

c:\Users\Innocent\OneDrive\Bureau\@ProgrammeAI_cours_session_04\Projet Capstone en IA\CityTaste\.venv\Scripts\python.exe
1.8.0
